In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ GOOGLE_API_KEY loaded successfully.")
except Exception as e:
    print("🔑 ERROR: Please add your 'GOOGLE_API_KEY' to Kaggle Secrets first.")
    print("   (Click Add-ons > Secrets > New Secret)")
    GOOGLE_API_KEY = None

✅ GOOGLE_API_KEY loaded successfully.


In [2]:
!pip install -q google-adk "uvicorn[standard]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.6/456.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.1/456.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<

In [3]:
!adk create backtester_agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/backtester_agent:
- .env
- __init__.py
- agent.py



In [4]:
%%writefile /kaggle/working/backtester_agent/agent.py

import os
from kaggle_secrets import UserSecretsClient
from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.a2a.utils.agent_to_a2a import to_a2a

# 1. Setup
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception:
    pass

retry_config = types.HttpRetryOptions(attempts=5)
model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)

# 2. Define the Backtester Agent
# It uses the Code Executor to actually run the math.
backtester = LlmAgent(
    model=model,
    name="BacktesterAgent",
    instruction="""You are a quantitative backtesting engine.
    You receive a trading strategy logic (e.g., 'Buy if price > 100').
    
    Your job is to:
    1. Generate Python code to simulate this strategy on dummy data.
    2. Calculate the 'Sharpe Ratio' and 'Max Drawdown'.
    3. Print the results clearly.
    
    You MUST use the Code Executor to calculate the numbers. Do not guess.""",
    code_executor=BuiltInCodeExecutor(),
)

# 3. Expose as A2A Service on Port 8001
app = to_a2a(backtester, port=8001)

Overwriting /kaggle/working/backtester_agent/agent.py


In [5]:
!adk create alpha_seeker_agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/alpha_seeker_agent:
- .env
- __init__.py
- agent.py



In [6]:
%%writefile /kaggle/working/alpha_seeker_agent/agent.py

import os
from kaggle_secrets import UserSecretsClient
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
    AGENT_CARD_WELL_KNOWN_PATH,
)

# --- Setup ---
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception:
    pass

model = Gemini(model="gemini-2.5-flash-lite", retry_options=types.HttpRetryOptions(attempts=5))

# --- Custom Tools ---

def fetch_market_data(ticker: str):
    """Fetches raw price data for a given ticker (Simulated)."""
    return f"Market Data for {ticker}: [150.0, 152.5, 149.0, 155.0, 160.0, 158.0, 162.0] (Trend: Generally Upward)"

def fetch_technical_indicators(ticker: str):
    """Calculates RSI and MACD for a given ticker (Simulated)."""
    return f"Indicators for {ticker}: RSI=75 (Overbought), MACD=Positive Divergence, Moving Average(50)=145.0"

def exit_loop():
    """Call this ONLY when the strategy is 'APPROVED' by the Risk Manager."""
    return {"status": "approved", "message": "Strategy passed risk checks."}

# --- Agents ---

# 1. The Strategist
strategist = LlmAgent(
    model=model,
    name="Strategist",
    instruction="""You are a hedge fund strategist.
    Based on the user's request, identify the Ticker Symbol we should trade.
    Output just the ticker symbol (e.g., 'AAPL').""",
    output_key="ticker",
)

# 2. The Quants (Parallel Team)
data_agent = LlmAgent(
    model=model,
    name="DataWorker",
    instruction="""Fetch raw price data for the ticker: {ticker}.""",
    tools=[FunctionTool(fetch_market_data)],
    output_key="price_data",
)

indicator_agent = LlmAgent(
    model=model,
    name="IndicatorWorker",
    instruction="""Fetch technical indicators for the ticker: {ticker}.""",
    tools=[FunctionTool(fetch_technical_indicators)],
    output_key="indicator_data",
)

# 3. The Risk Loop Agents

remote_backtester = RemoteA2aAgent(
    name="backtester_agent",
    description="Runs a backtest simulation and returns Sharpe Ratio and Drawdown.",
    agent_card=f"http://127.0.0.1:8001{AGENT_CARD_WELL_KNOWN_PATH}",
)

strategy_developer = LlmAgent(
    model=model,
    name="StrategyDev",
    instruction="""You are a strategy developer.
    
    Your Goal: Create a specific trading rule (e.g., "Buy when RSI > 70").
    
    Use the `backtester_agent` tool to test your rule and get the numbers.
    
    If the Risk Manager has rejected a previous attempt in the chat history,
    adjust your rule to be safer (e.g., lower position size, tighter stops) and test again.""",
    output_key="backtest_results",
    sub_agents=[remote_backtester] 
)

# --- FIXED INSTRUCTION ---
# Removed {backtest_results} variable. The agent will read the history instead.
risk_manager = LlmAgent(
    model=model,
    name="RiskManager",
    instruction="""You are a strict Risk Manager.
    Look at the most recent backtest results in the conversation.
    
    - If 'Max Drawdown' is greater than 15% (0.15), respond: "REJECTED: Drawdown too high."
    - Otherwise, respond: "APPROVED" """,
    output_key="risk_feedback",
)

# --- FIXED INSTRUCTION ---
# Removed {risk_feedback} variable.
approval_agent = LlmAgent(
    model=model,
    name="ApprovalCheck",
    instruction="""Check the Risk Manager's decision in the conversation.
    - If they said "APPROVED", call the `exit_loop` tool.
    - If they said "REJECTED", output "Refining strategy..." to continue the loop.""",
    tools=[FunctionTool(exit_loop)]
)

# --- Workflow Assembly ---

quant_team = ParallelAgent(
    name="QuantTeam",
    sub_agents=[data_agent, indicator_agent]
)

development_loop = LoopAgent(
    name="StrategyLoop",
    sub_agents=[strategy_developer, risk_manager, approval_agent],
    max_iterations=3
)

root_agent = SequentialAgent(
    name="AlphaSeekerPipeline",
    sub_agents=[strategist, quant_team, development_loop]
)

print("✅ Fixed 'alpha_seeker_agent/agent.py' written successfully.")

Overwriting /kaggle/working/alpha_seeker_agent/agent.py


In [7]:
import subprocess
import time
import requests

print("🚀 Starting Server 1 (Backtester on port 8001)...")

server_process = subprocess.Popen(
    [
        "uvicorn",
        "agent:app",
        "--host", "127.0.0.1",
        "--port", "8001",
        "--app-dir", "/kaggle/working/backtester_agent/"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},
)

print("...Waiting 30 seconds for server to boot...")
time.sleep(30)

try:
    response = requests.get("http://127.0.0.1:8001/.well-known/agent-card.json")
    if response.status_code == 200:
        print(" SUCCESS: Backtester is online.")
    else:
        print(f"❌ FAILED: Status {response.status_code}")
        print(server_process.stderr.read().decode())
except Exception as e:
    print(f"❌ FAILED: {e}")

🚀 Starting Server 1 (Backtester on port 8001)...
...Waiting 30 seconds for server to boot...
 SUCCESS: Backtester is online.


In [8]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"
    servers = list(list_running_servers())
    if not servers: raise Exception("No servers found.")
    baseURL = servers[0]["base_url"]
    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError: raise Exception(f"Could not parse URL: {baseURL}")
    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"
    display(HTML(f"<a href='{url}' target='_blank'>Open ADK Web UI ↗</a>"))
    return url_prefix

url_prefix = get_adk_proxy_url()

Exception: No servers found.

In [ ]:
!adk web --url_prefix {url_prefix} \
         --session_service_uri "sqlite:////kaggle/working/alpha_chat.db" \
         --log_level DEBUG